# 06 — Scenario generation and ADOPT

## Question

How do we discretize the empirical (Δd, ΔE) joint into K=8 scenarios, and how do we compute the ADOPT scaling γ?

## Why this test exists

F1 is a scenario-averaged QUBO. The scenarios come from a k-means clustering on the calibration joint (Δd, ΔE), with K frozen at 8. ADOPT is a single-scalar pre-solve adaptive mechanism: `γ = 1 + α · mean(σ_i / R̄_i)`, with α frozen at 1.0. Both K and α are pre-registered BEFORE any held-out use.

## Method

1. **Standardize** the calibration (Δd, ΔE) using the calibration mean    and standard deviation. Apply the same transformation to all    scenarios.
2. **K-means** with K=8, seed `20260829 + 8 = 20260837`. The seed is    frozen. K=4 and K=16 are computed as diagnostic sensitivity only;    K=8 is the official.
3. **γ** = 1 + α · mean(σ_i / R̄_i) on the calibration joint, with    α=1.0. The ratio σ_i / R̄_i is the coefficient-of-variation of the    effective number of charging slots required per EV in scenario ω.

**FROZEN CONFIGURATION.** The methodology is frozen at `artifacts/final_experiment_config.json` (version `stage7.v1`). K=8, α=1.0, M_window=1e6, ρ_d=1.0, ρ_p=0.1, ρ_cap=0.5, P_target=6.6 kW, P_site_max=9.9 kW, Δ=15 min, calibration window 2018-05-01..2019-07-01, held-out window 2019-07-01..2020-01-01, QAOA p=1 / COBYLA / seeds [0,1,2] / shots 1024. No parameter may be modified based on held-out results.


## Implementation


In [ ]:
import sys, pathlib, json
import numpy as np
sys.path.insert(0, str(pathlib.Path('..').resolve()))
from stage5.uncertainty import (placeholder_uncertainty, kmeans_joint,
                                   compute_robust_rho_d, ADOPTParameters)
from stage3.ev_scheduling import toy_instance

# NOTE: placeholder until real data. Real K=8 + γ computed in notebook 10.
samples, _ = placeholder_uncertainty(seed=20260829)
cal = [s for s in samples if s.calibration]
dd = np.array([s.delta_d_minutes for s in cal if s.delta_d_minutes is not None])
de = np.array([s.delta_e_kwh for s in cal if s.delta_e_kwh is not None])
K = 8
res = kmeans_joint(dd, de, K=K, seed=20260829 + K)
print(f"K-means K={K}, seed=20260829+8")
for s in res['clusters']:
    print(f"  cluster {s['cluster']}: weight={s['weight']:.4f}  "
          f"centroid=({s['centroid_delta_d_minutes']:.2f} min, "
          f"{s['centroid_delta_e_kwh']:.3f} kWh)")
print()
inst = toy_instance('toy_B_3x4', N=3, T=4)
rho_d_robust, adopt_stats = compute_robust_rho_d(1.0, inst, cal, alpha=1.0)
print(f"ADOPT γ (placeholder): {adopt_stats['gamma']:.4f}")
print(f"ADOPT ρ_d_robust:      {rho_d_robust:.4f}")
print()
print(f"This γ is computed on the PLACEHOLDER calibration joint.")
print(f"The real γ is computed in notebook 10 from the live ACN-Data calibration joint.")


## Result (placeholder)

On the placeholder, the K=8 k-means produces a dominant cluster (weight ≈ 0.74) at a small-positive Δd and small-negative ΔE. The ADOPT γ on the placeholder is ≈ 1.64 (rounded to 1.6414 in the Stage 5 doc).

## Interpretation

The scenario engine and ADOPT are pre-registered (K=8, α=1.0). The γ value is computed on calibration only and frozen before any held-out use. The placeholder γ is not the ACN-Data γ.

The dominant cluster on the placeholder is a *property of the synthetic mixture* (which is over-delivery-skewed). On real ACN-Data, the cluster distribution will differ, and so will γ. The framework absorbs this difference automatically — the only thing that must not change is K and α.

## Limitations

- The placeholder γ is not the real γ. The real γ is reported in   notebook 10 once the live calibration joint is available.
- γ is a single scalar; it cannot represent full distributional   change between calibration and held-out. The distribution-shift   diagnostic (Case A / B / C) is the right tool for that.


## Quantum-advantage disclaimer

This work does **not** claim quantum advantage. The 11-qubit instance is small enough that the exact classical optimum is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. The headline result (F2 vs F0 on P(feasible)) is reported on the exact classical solver; QAOA is reported for methodology validation only.


## No post-hoc tuning

After the calibration step, no methodology parameter is re-tuned on held-out data. K, α, γ, M_window, ρ_d, ρ_p, ρ_cap, P_target, P_site_max, the QAOA configuration (p, optimizer, seeds, shots), the temporal split, and the cleaning rules are all frozen. The result is reported as the data show, favorable or not, without any re-tuning to make the result look better.


## Reproducibility

Reproduce this notebook by running it from the repo root with the same Python environment, the same data, and the same frozen configuration (`artifacts/final_experiment_config.json`, version `stage7.v1`). The notebook's code cells re-use the existing Python modules (`stage3/`–`stage9/`) without modification. See `notebooks/README.md` for the per-notebook contract.


## Quantum-advantage disclaimer

This work does **not** claim quantum advantage. The 11-qubit instance is small enough that the exact classical optimum is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. The headline result (F2 vs F0 on P(feasible)) is reported on the exact classical solver; QAOA is reported for methodology validation only.


## No post-hoc tuning

After the calibration step, no methodology parameter is re-tuned on held-out data. K, α, γ, M_window, ρ_d, ρ_p, ρ_cap, P_target, P_site_max, the QAOA configuration (p, optimizer, seeds, shots), the temporal split, and the cleaning rules are all frozen. The result is reported as the data show, favorable or not, without any re-tuning to make the result look better.


## Reproducibility

Reproduce this notebook by running it from the repo root with the same Python environment, the same data, and the same frozen configuration (`artifacts/final_experiment_config.json`, version `stage7.v1`). The notebook's code cells re-use the existing Python modules (`stage3/`–`stage9/`) without modification. See `notebooks/README.md` for the per-notebook contract.
